In [1]:
# =====================================================
# IMPORTS
# =====================================================
import numpy as np
import pandas as pd
import os

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    accuracy_score,
    classification_report
)

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    StackingClassifier,
    AdaBoostClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier


# =====================================================
# 1. CARGAR DATOS
# =====================================================
def cargar_y_preparar_datos(ruta_archivo):
    df = pd.read_excel(ruta_archivo)

    df_viv = df[df['Proposito'].astype(str)
                .str.contains('Vivienda', case=False, na=False)].copy()

    df_viv['Impago_Label'] = df_viv['Impago'].map({0: 0, 1: 1})

    return df_viv


ruta_real = os.path.join('..', 'Datos', 'Originales', 'información_préstamos.xlsx')
df = cargar_y_preparar_datos(ruta_real)

target_col = "Impago_Label"

y = df[target_col]
X = df.drop(columns=[target_col])

# eliminar fuga
if "Impago" in X.columns:
    X = X.drop(columns=["Impago"])

# eliminar IDs
for col in X.columns:
    if "id" in col.lower():
        X = X.drop(columns=[col])

# eliminar alta cardinalidad
high_card_cols = [col for col in X.columns if X[col].nunique() > 50]
X = X.drop(columns=high_card_cols)

# one hot
cat_cols = X.select_dtypes(include="object").columns
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X = X.astype("float32")


# =====================================================
# 2. SPLIT ANTES DEL CLUSTERING
# =====================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)


# =====================================================
# 3. CLUSTERING SOLO EN TRAIN
# =====================================================
scaler_cluster = StandardScaler()
X_train_cluster = scaler_cluster.fit_transform(X_train)
X_test_cluster = scaler_cluster.transform(X_test)

def evaluar_clustering(X, labels):
    if len(set(labels)) < 2:
        return -1
    return silhouette_score(X, labels)

# KMeans
kmeans = KMeans(n_clusters=3, random_state=42)
labels_k_train = kmeans.fit_predict(X_train_cluster)
labels_k_test = kmeans.predict(X_test_cluster)
score_k = evaluar_clustering(X_train_cluster, labels_k_train)

# Hierarchical
hc = AgglomerativeClustering(n_clusters=3)
labels_h_train = hc.fit_predict(X_train_cluster)
score_h = evaluar_clustering(X_train_cluster, labels_h_train)
labels_h_test = kmeans.predict(X_test_cluster)

# DBSCAN
db = DBSCAN(eps=2, min_samples=10)
labels_d_train = db.fit_predict(X_train_cluster)
score_d = evaluar_clustering(X_train_cluster, labels_d_train)
labels_d_test = np.zeros(len(X_test))

scores = {"KMeans": score_k,
          "Hierarchical": score_h,
          "DBSCAN": score_d}

best_cluster = max(scores, key=scores.get)
print("Mejor clustering:", best_cluster)

if best_cluster == "KMeans":
    X_train["cluster_feature"] = labels_k_train
    X_test["cluster_feature"] = labels_k_test
elif best_cluster == "Hierarchical":
    X_train["cluster_feature"] = labels_h_train
    X_test["cluster_feature"] = labels_h_test
else:
    X_train["cluster_feature"] = labels_d_train
    X_test["cluster_feature"] = labels_d_test


# =====================================================
# 4. FUNCIÓN ENTRENAMIENTO
# =====================================================
def entrenar_modelo(
    nombre_modelo,
    modelo,
    param_grid,
    X_train, X_test,
    y_train, y_test,
    usar_smote=False,
    usar_pca=False
):

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    y_train_res = y_train.copy()

    if usar_smote:
        sm = SMOTE(random_state=42)
        X_train_scaled, y_train_res = sm.fit_resample(
            X_train_scaled, y_train
        )

    if usar_pca:
        pca = PCA(n_components=0.95, random_state=42)
        X_train_scaled = pca.fit_transform(X_train_scaled)
        X_test_scaled = pca.transform(X_test_scaled)

    grid = GridSearchCV(
        modelo,
        param_grid,
        cv=3,
        scoring="accuracy",
        n_jobs=-1
    )

    grid.fit(X_train_scaled, y_train_res)

    y_pred = grid.best_estimator_.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)

    print("\n", "="*60)
    print(f"{nombre_modelo} | SMOTE={usar_smote} | PCA={usar_pca}")
    print("Best params:", grid.best_params_)
    print("Accuracy:", round(acc,4))
    print(classification_report(y_test, y_pred))

    return {
        "Modelo": nombre_modelo,
        "SMOTE": usar_smote,
        "PCA": usar_pca,
        "Accuracy": acc
    }


# =====================================================
# 5. MODELOS
# =====================================================
modelos = {

    "LogReg": (
        LogisticRegression(max_iter=1000),
        {"C": [0.01, 0.1, 1]}
    ),

    "RandomForest": (
        RandomForestClassifier(random_state=42),
        {"n_estimators": [100, 200]}
    ),

    "DecisionTree": (
        DecisionTreeClassifier(random_state=42),
        {"max_depth": [None, 5, 10]}
    ),

    "AdaBoost": (
        AdaBoostClassifier(random_state=42),
        {"n_estimators": [50, 100]}
    ),

    "XGBoost": (
        XGBClassifier(eval_metric="logloss", random_state=42),
        {"n_estimators": [100],
         "max_depth": [3, 6]}
    )
}

estimadores_base = [
    ("rf", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("dt", DecisionTreeClassifier(random_state=42)),
    ("nb", GaussianNB())
]

stacking = StackingClassifier(
    estimators=estimadores_base,
    final_estimator=LogisticRegression()
)

modelos["Stacking"] = (
    stacking,
    {"final_estimator__C": [0.1, 1]}
)


# =====================================================
# 6. EJECUTAR Y GUARDAR RESULTADOS
# =====================================================
combinaciones = [
    (False, False),
    (True, False),
    (False, True),
    (True, True)
]

resultados_finales = []

for nombre, (modelo, grid) in modelos.items():
    for smote_flag, pca_flag in combinaciones:

        res = entrenar_modelo(
            nombre,
            modelo,
            grid,
            X_train, X_test,
            y_train, y_test,
            smote_flag,
            pca_flag
        )

        resultados_finales.append(res)

df_resultados = pd.DataFrame(resultados_finales)

Mejor clustering: DBSCAN

LogReg | SMOTE=False | PCA=False
Best params: {'C': 0.01}
Accuracy: 0.8977
              precision    recall  f1-score   support

           0       0.90      1.00      0.95     11510
           1       0.00      0.00      0.00      1312

    accuracy                           0.90     12822
   macro avg       0.45      0.50      0.47     12822
weighted avg       0.81      0.90      0.85     12822



c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}


LogReg | SMOTE=True | PCA=False
Best params: {'C': 0.1}
Accuracy: 0.5648
              precision    recall  f1-score   support

           0       0.92      0.57      0.70     11510
           1       0.13      0.54      0.20      1312

    accuracy                           0.56     12822
   macro avg       0.52      0.56      0.45     12822
weighted avg       0.84      0.56      0.65     12822


LogReg | SMOTE=False | PCA=True
Best params: {'C': 0.01}
Accuracy: 0.8977
              precision    recall  f1-score   support

           0       0.90      1.00      0.95     11510
           1       0.00      0.00      0.00      1312

    accuracy                           0.90     12822
   macro avg       0.45      0.50      0.47     12822
weighted avg       0.81      0.90      0.85     12822



c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}


LogReg | SMOTE=True | PCA=True
Best params: {'C': 0.01}
Accuracy: 0.5508
              precision    recall  f1-score   support

           0       0.92      0.55      0.69     11510
           1       0.12      0.55      0.20      1312

    accuracy                           0.55     12822
   macro avg       0.52      0.55      0.44     12822
weighted avg       0.83      0.55      0.64     12822


RandomForest | SMOTE=False | PCA=False
Best params: {'n_estimators': 200}
Accuracy: 0.8967
              precision    recall  f1-score   support

           0       0.90      1.00      0.95     11510
           1       0.07      0.00      0.00      1312

    accuracy                           0.90     12822
   macro avg       0.48      0.50      0.47     12822
weighted avg       0.81      0.90      0.85     12822


RandomForest | SMOTE=True | PCA=False
Best params: {'n_estimators': 200}
Accuracy: 0.8904
              precision    recall  f1-score   support

           0       0.90      0.99 

...


XGBoost | SMOTE=True | PCA=False
Best params: {'max_depth': 6, 'n_estimators': 100}
Accuracy: 0.8885
              precision    recall  f1-score   support

           0       0.90      0.99      0.94     11510
           1       0.10      0.01      0.02      1312

    accuracy                           0.89     12822
   macro avg       0.50      0.50      0.48     12822
weighted avg       0.82      0.89      0.85     12822


XGBoost | SMOTE=False | PCA=True
Best params: {'max_depth': 3, 'n_estimators': 100}
Accuracy: 0.8976
              precision    recall  f1-score   support

           0       0.90      1.00      0.95     11510
           1       0.00      0.00      0.00      1312

    accuracy                           0.90     12822
   macro avg       0.45      0.50      0.47     12822
weighted avg       0.81      0.90      0.85     12822


XGBoost | SMOTE=True | PCA=True
Best params: {'max_depth': 6, 'n_estimators': 100}
Accuracy: 0.7163
              precision    recall  f1-sco

c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}


Stacking | SMOTE=True | PCA=False
Best params: {'final_estimator__C': 0.1}
Accuracy: 0.8244
              precision    recall  f1-score   support

           0       0.90      0.91      0.90     11510
           1       0.12      0.12      0.12      1312

    accuracy                           0.82     12822
   macro avg       0.51      0.51      0.51     12822
weighted avg       0.82      0.82      0.82     12822


Stacking | SMOTE=False | PCA=True
Best params: {'final_estimator__C': 0.1}
Accuracy: 0.8977
              precision    recall  f1-score   support

           0       0.90      1.00      0.95     11510
           1       0.00      0.00      0.00      1312

    accuracy                           0.90     12822
   macro avg       0.45      0.50      0.47     12822
weighted avg       0.81      0.90      0.85     12822



c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}


Stacking | SMOTE=True | PCA=True
Best params: {'final_estimator__C': 1}
Accuracy: 0.4563
              precision    recall  f1-score   support

           0       0.90      0.44      0.59     11510
           1       0.11      0.58      0.18      1312

    accuracy                           0.46     12822
   macro avg       0.50      0.51      0.39     12822
weighted avg       0.82      0.46      0.55     12822



In [2]:
print("\n=========== RESULTADOS FINALES ===========")
print(df_resultados.sort_values("Accuracy", ascending=False))


=========== RESULTADOS FINALES ===========
          Modelo  SMOTE    PCA  Accuracy
0         LogReg  False  False  0.897676
8   DecisionTree  False  False  0.897676
22      Stacking  False   True  0.897676
20      Stacking  False  False  0.897676
16       XGBoost  False  False  0.897676
14      AdaBoost  False   True  0.897676
10  DecisionTree  False   True  0.897676
12      AdaBoost  False  False  0.897676
2         LogReg  False   True  0.897676
18       XGBoost  False   True  0.897598
6   RandomForest  False   True  0.897364
4   RandomForest  False  False  0.896662
13      AdaBoost   True  False  0.895414
5   RandomForest   True  False  0.890423
17       XGBoost   True  False  0.888473
21      Stacking   True  False  0.824364
9   DecisionTree   True  False  0.797380
19       XGBoost   True   True  0.716347
7   RandomForest   True   True  0.686554
15      AdaBoost   True   True  0.593433
1         LogReg   True  False  0.564810
3         LogReg   True   True  0.550772
11  DecisionT